In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


ModuleNotFoundError: No module named 'sklearn'

In [5]:
INPUT_CSV = "sales_sample.csv"         # ensure this file is in the same folder
OUTPUT_PRED_CSV = "predictions.csv"
TEST_WINDOW_DAYS = 90

In [6]:
if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(f"Could not find {INPUT_CSV} in the current folder. Put the csv there and re-run.")
df = pd.read_csv(INPUT_CSV, parse_dates=["Date"])
df.sort_values(["Store", "Date"], inplace=True)
df.reset_index(drop=True, inplace=True)



In [7]:
print("Loaded data shape:", df.shape)
print("Columns:", df.columns.tolist())


Loaded data shape: (7310, 9)
Columns: ['Date', 'Store', 'StoreType', 'Sales', 'Customers', 'Promo', 'Holiday', 'DayOfWeek', 'CompetitionDistance']


In [8]:
print("\nSample rows:")
print(df.head().to_string())


Sample rows:
        Date Store StoreType   Sales  Customers  Promo  Holiday  DayOfWeek  CompetitionDistance
0 2023-01-01   S01         C  282.83         12      0        0          6               1149.0
1 2023-01-02   S01         C  306.75         15      0        0          0                819.5
2 2023-01-03   S01         C  256.76         17      0        0          1                856.2
3 2023-01-04   S01         C  356.10         21      1        0          2               1029.1
4 2023-01-05   S01         C  219.44          8      0        0          3               1237.3


In [9]:
print("\nSample rows:")
print(df.tail().to_string())


Sample rows:
           Date Store StoreType   Sales  Customers  Promo  Holiday  DayOfWeek  CompetitionDistance
7305 2024-12-27   S10         A  173.28          7      0        0          4               1172.2
7306 2024-12-28   S10         A  274.67         10      0        0          5               1110.8
7307 2024-12-29   S10         A  185.85          9      0        0          6                925.6
7308 2024-12-30   S10         A  325.22         16      1        0          0                847.8
7309 2024-12-31   S10         A  262.97         10      0        0          1                751.2


In [10]:
# 2) Basic feature engineering (time features)
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)

In [11]:
# 3) Lags and rolling stats (per Store)
def add_lag_roll(group):
    grp = group.sort_values("Date").copy()
    grp["lag_1"] = grp["Sales"].shift(1)
    grp["lag_7"] = grp["Sales"].shift(7)
    grp["roll_7_mean"] = grp["Sales"].shift(1).rolling(window=7, min_periods=1).mean()
    grp["roll_30_mean"] = grp["Sales"].shift(1).rolling(window=30, min_periods=1).mean()
    return grp

df = df.groupby("Store").apply(add_lag_roll).reset_index(drop=True)

C:\Users\ksnlp\AppData\Local\Temp\ipykernel_20056\1439899116.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("Store").apply(add_lag_roll).reset_index(drop=True)


In [12]:
# Fill missing lag/roll values with store median, fallback to global median
for c in ["lag_1", "lag_7", "roll_7_mean", "roll_30_mean"]:
    df[c] = df.groupby("Store")[c].transform(lambda x: x.fillna(x.median()))
    df[c].fillna(df[c].median(), inplace=True)

C:\Users\ksnlp\AppData\Local\Temp\ipykernel_20056\1634677559.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[c].fillna(df[c].median(), inplace=True)
C:\Users\ksnlp\AppData\Local\Temp\ipykernel_20056\1634677559.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, 

In [13]:
# 4) Categorical encoding: StoreType -> dummies (drop_first to avoid multicollinearity)
df = pd.get_dummies(df, columns=["StoreType"], prefix="StoreType", drop_first=True)

In [14]:
# 5) Select features and target
feature_cols = [
    "Customers", "Promo", "Holiday", "DayOfWeek", "CompetitionDistance",
    "Year", "Month", "WeekOfYear", "IsWeekend",
    "lag_1", "lag_7", "roll_7_mean", "roll_30_mean"
]

In [16]:
# add any StoreType dummies present
feature_cols += [c for c in df.columns if c.startswith("StoreType_")]
feature_cols = [c for c in feature_cols if c in df.columns]
target_col = "Sales"

print("\nUsing features:", feature_cols)


Using features: ['Customers', 'Promo', 'Holiday', 'DayOfWeek', 'CompetitionDistance', 'Year', 'Month', 'WeekOfYear', 'IsWeekend', 'lag_1', 'lag_7', 'roll_7_mean', 'roll_30_mean', 'StoreType_B', 'StoreType_C', 'StoreType_B', 'StoreType_C']


In [17]:
# 6) Time-based train/test split
cutoff_date = df["Date"].max() - pd.Timedelta(days=TEST_WINDOW_DAYS)
train_df = df[df["Date"] <= cutoff_date].copy()
test_df = df[df["Date"] > cutoff_date].copy()
print(f"\nCutoff date for train/test split: {cutoff_date.date()}")
print("Train rows:", train_df.shape[0], "Test rows:", test_df.shape[0])

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]


Cutoff date for train/test split: 2024-10-02
Train rows: 6410 Test rows: 900


In [19]:
# 7) Model Selection
try:
    from xgboost import XGBRegressor
    model = XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
    model_name = "XGBoost"
except:
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model_name = "RandomForest"

print("\nTraining Model:", model_name)
model.fit(X_train, y_train)



Training Model: XGBoost


AttributeError: 'DataFrame' object has no attribute 'dtype'